In [ ]:
"""
实验三：双平台动态博弈实验
"""

import os
import re
import json
import random
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

# ================= 配置区 =================
MODEL_PATH = "Qwen/Qwen2.5-14B-Instruct"
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/eco-simp"
MEMORY_DIR = os.path.join(BASE_DIR, "exp3_memories_v4")
RESULTS_DIR = os.path.join(BASE_DIR, "exp3_results_v4")
MAX_MEMORY_LINES = 30
MAX_ROUNDS = 10
EXIT_LOSE_ROUNDS = 2
COOLDOWN_ROUNDS = 2
CONVERGENCE_THRESHOLD = 0.01
CONVERGENCE_ROUNDS = 3

# 全局随机种子
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# 平台策略约束范围
A_COMM_RANGE = (0.06, 0.14)
A_SUB_RANGE = (0.01, 0.05)
B_COMM_RANGE = (0.12, 0.21)
B_SUB_RANGE = (0.04, 0.08)

# 4种初始策略组合
INIT_STRATEGIES = [
    {"A_comm": 0.06, "A_sub": 0.01, "B_comm": 0.12, "B_sub": 0.04},
    {"A_comm": 0.06, "A_sub": 0.01, "B_comm": 0.21, "B_sub": 0.08},
    {"A_comm": 0.14, "A_sub": 0.05, "B_comm": 0.12, "B_sub": 0.04},
    {"A_comm": 0.14, "A_sub": 0.05, "B_comm": 0.21, "B_sub": 0.08},
]

os.makedirs(RESULTS_DIR, exist_ok=True)

# ================= 1. 模型加载 =================
print(f"加载模型: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)
print("模型加载完成")


def generate_llm_response(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=True, temperature=0.75, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return resp.strip()


# ================= 2. Agent 基类 =================
class BaseAgent:
    def __init__(self, agent_id, role_type, desc):
        self.agent_id = agent_id
        self.role_type = role_type
        self.description = desc
        self.memory_file = ""

    def init_memory(self):
        with open(self.memory_file, "w", encoding="utf-8") as f:
            f.write(f"【角色】{self.role_type}｜{self.description}\n【历史】\n")

    def read_memory(self):
        if not os.path.exists(self.memory_file):
            return ""
        with open(self.memory_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) <= MAX_MEMORY_LINES + 1:
            return "".join(lines)
        return "".join(lines[:1] + lines[-MAX_MEMORY_LINES:])

    def append_memory(self, text):
        with open(self.memory_file, "a", encoding="utf-8") as f:
            f.write(text + "\n")

    def decide(self, context, task):
        memory = self.read_memory()
        prompt = self.build_prompt(memory, context, task)
        res = generate_llm_response(prompt)
        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{context[:80]}...｜决策：{decision}")
        return decision, thinking

    def build_prompt(self, memory, context, task):
        return ""


# ================= 3. 平台Agent =================
class PlatformAgent(BaseAgent):
    def __init__(self, plat_type, comm_range, sub_range):
        if plat_type == "A":
            desc = "内容驱动型电商平台，以内容推荐与场景化种草为核心"
        else:
            desc = "搜索货架型电商平台，以搜索比价为核心"
        super().__init__(f"Platform_{plat_type}", f"{plat_type}平台", desc)
        self.plat_type = plat_type
        self.comm_range = comm_range
        self.sub_range = sub_range
        self.current_comm = 0.0
        self.current_sub = 0.0
        self.last_revenue = 0.0
        self.last_gmv = 0.0
        self.last_active_sellers = 0
        self.last_exited_sellers = 0
        self.last_transactions = 0
        self.last_market_share = 0.5
        self.last_inf_count = 0

    def reset(self, init_comm, init_sub):
        self.current_comm = init_comm
        self.current_sub = init_sub
        self.last_revenue = 0.0
        self.last_gmv = 0.0
        self.last_active_sellers = 0
        self.last_exited_sellers = 0
        self.last_transactions = 0
        self.last_market_share = 0.5
        self.last_inf_count = 0

    def build_prompt(self, memory, context, task):
        if self.plat_type == "A":
            identity = """你是【内容驱动型电商平台A】的策略决策者。

【平台定位】
- 以内容推荐与场景化种草为核心驱动力
- 达人生态是你的核心竞争优势，达人推荐是用户购买的重要驱动
- 用户购物体验更社交化、更具发现感

【盈利模式】
- 收入来源：佣金收入 + 广告收入
- 补贴支出：平台出钱补贴用户（降低用户实付价格以拉动销量）
- 平台营收 = 佣金收入 + 广告收入 - 补贴支出

【策略逻辑】
- 降低佣金→吸引更多商家入驻→供给丰富→用户体验好→GMV增长，但单笔佣金收入减少
- 提高补贴→用户实付更低→刺激购买→GMV增长，但补贴支出增加侵蚀利润
- 需要在商家留存、用户活跃、平台利润之间找到平衡
- 关注商家退出率：佣金过高会导致商家利润压缩→退出平台→供给萎缩
- 关注达人留存：达人倾向于在你的平台活跃，这是相对B平台的优势"""
        else:
            identity = """你是【搜索货架型电商平台B】的策略决策者。

【平台定位】
- 以搜索比价为核心，用户带着明确购买意图来平台
- 达人影响力相对较弱，用户更依赖搜索和商品排名
- 用户购买转化率高，但获客依赖价格竞争力

【盈利模式】
- 收入来源：佣金收入（主要）+ 广告收入
- 补贴支出：平台出钱补贴用户
- 平台营收 = 佣金收入 + 广告收入 - 补贴支出

【策略逻辑】
- 佣金率通常高于A平台（因用户购买意图明确，商家转化率高，商家愿意承担更高佣金）
- 高补贴可以撬动消费、抵消商家高佣金带来的价格劣势
- 需要在高佣金带来的收入与商家承受能力之间平衡
- 关注与A平台的价格竞争力：用户可能因补贴差异在两平台间切换
- 你的劣势是达人生态弱，需要靠价格和补贴弥补"""

        return f"""{identity}

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先分析上轮表现、竞争态势和生态健康度，然后给出策略调整。
调整幅度建议不超过0.02（避免剧烈波动），除非当前策略明显失败。
<thinking>你的竞争分析（2-3句话）</thinking>
<decision>佣金:X,补贴:Y</decision>"""
    def decide_strategy(self, opponent_comm, opponent_sub, opponent_gmv, opponent_revenue, round_num):
        """平台自主决策策略"""
        # 判断上轮表现趋势
        if self.last_revenue > 0:
            rev_desc = f"￥{self.last_revenue:.0f}"
        else:
            rev_desc = f"￥{self.last_revenue:.0f}（亏损）"

        share_desc = "领先" if self.last_market_share > 0.5 else ("持平" if self.last_market_share == 0.5 else "落后")

        ctx = (f"当前第{round_num}轮。\n"
               f"【你的上轮表现】\n"
               f"  策略：佣金={self.current_comm:.2%}，补贴={self.current_sub:.2%}\n"
               f"  GMV=￥{self.last_gmv:.0f}，营收={rev_desc}，市场份额={self.last_market_share:.1%}（{share_desc}）\n"
               f"  活跃商家={self.last_active_sellers}家，退出商家={self.last_exited_sellers}家，交易={self.last_transactions}笔\n"
               f"  {'达人在你平台=' + str(self.last_inf_count) + '人' if self.plat_type == 'A' else ''}\n"
               f"【对手平台上轮表现】\n"
               f"  策略：佣金={opponent_comm:.2%}，补贴={opponent_sub:.2%}\n"
               f"  GMV=￥{opponent_gmv:.0f}，营收=￥{opponent_revenue:.0f}")

        task = (f"请在约束范围内调整你的佣金率和补贴率：\n"
                f"  佣金率范围：{self.comm_range[0]} ~ {self.comm_range[1]}（当前={self.current_comm}）\n"
                f"  补贴率范围：{self.sub_range[0]} ~ {self.sub_range[1]}（当前={self.current_sub}）\n"
                f"请必须输出纯小数（例如0.08，绝对不要带百分号）。格式：<decision>佣金:0.XX,补贴:0.YY</decision>")

        dec, thinking = self.decide(ctx, task)

        new_comm = self.current_comm
        new_sub = self.current_sub

        comm_m = re.search(r'佣金[:：]?\s*(\d+\.?\d*)\%?', dec)
        sub_m = re.search(r'补贴[:：]?\s*(\d+\.?\d*)\%?', dec)

        if comm_m:
            val = float(comm_m.group(1))
            if val >= 1.0: val = val / 100.0  # 如果 LLM 输出了 8 或 8.00，自动转为 0.08
            new_comm = max(self.comm_range[0], min(self.comm_range[1], val))

        if sub_m:
            val = float(sub_m.group(1))
            if val >= 1.0: val = val / 100.0
            new_sub = max(self.sub_range[0], min(self.sub_range[1], val))

        old_comm, old_sub = self.current_comm, self.current_sub
        self.current_comm = round(new_comm, 4)
        self.current_sub = round(new_sub, 4)

        return {
            "commission": self.current_comm, "subsidy": self.current_sub,
            "old_commission": old_comm, "old_subsidy": old_sub,
            "thinking": thinking
        }


# ================= 4. 商家Agent =================
SELLER_CONFIGS = [
    ("Seller_L1", "低价走量", 30, 45, 55, 0.00, 3, "保守求生，倾向低价低广告，追求稳定出单"),
    ("Seller_L2", "低价走量", 30, 45, 55, 0.00, 3, "薄利多销，愿意在定价区间中段定价换取更多销量"),
    ("Seller_L3", "低价走量", 30, 45, 55, 0.00, 3, "轻微卷价，喜欢比同行低1-2元抢单"),
    ("Seller_L4", "低价走量", 30, 45, 55, 0.00, 3, "广告厌恶型，几乎不投广告，依靠自然流量"),
    ("Seller_L5", "低价走量", 30, 45, 55, 0.00, 3, "跟随市场，参考上轮结果调整策略"),
    ("Seller_L6", "低价走量", 30, 45, 55, 0.00, 3, "激进投放型，愿意在低价品上投满广告博曝光"),
    ("Seller_M1", "中端品质", 50, 130, 170, 0.05, 15, "均衡稳健，追求利润与销量的平衡点"),
    ("Seller_M2", "中端品质", 50, 130, 170, 0.05, 15, "品牌导向，倾向高定价维护品牌调性，广告适度"),
    ("Seller_M3", "中端品质", 50, 130, 170, 0.05, 15, "转化优先，愿意降价提升转化率，广告投入较高"),
    ("Seller_H1", "高端溢价", 100, 270, 330, 0.10, 45, "高端长期主义，坚持高定价，重品牌形象和复购"),
]


class SellerAgent(BaseAgent):
    def __init__(self, sid, seller_type, cost, price_min, price_max, margin_safe, ad_max, character):
        super().__init__(sid, seller_type, f"{seller_type}｜{character}")
        self.seller_type = seller_type
        self.unit_cost = cost
        self.price_min = price_min
        self.price_max = price_max
        self.margin_safe = margin_safe
        self.ad_max = ad_max
        self.character = character
        self.last_margin_A = 0.0
        self.last_margin_B = 0.0
        self.last_sales_A = 0
        self.last_sales_B = 0
        self.consecutive_lose_A = 0
        self.consecutive_lose_B = 0
        self.exited_A = False
        self.exited_B = False
        self.cooldown_A = 0
        self.cooldown_B = 0

    def reset(self):
        self.last_margin_A = 0.0
        self.last_margin_B = 0.0
        self.last_sales_A = 0
        self.last_sales_B = 0
        self.consecutive_lose_A = 0
        self.consecutive_lose_B = 0
        self.exited_A = False
        self.exited_B = False
        self.cooldown_A = 0
        self.cooldown_B = 0

    def build_prompt(self, memory, context, task):
        margin_pct = f"{self.margin_safe:.0%}" if self.margin_safe > 0 else "0%（可微利生存）"
        return f"""你是一个同时在两个电商平台（A和B）经营的商家，需要根据双平台政策和市场环境做出理性的经营决策。

【商家身份】
- 类型：{self.seller_type}
- 经营性格：{self.character}
- 单位生产成本：￥{self.unit_cost}
- 允许定价区间：￥{self.price_min} ~ ￥{self.price_max}（两平台同价，必须在此区间内）
- 每个平台广告投入上限：￥{self.ad_max}（0到{self.ad_max}之间）
- 毛利安全线：{margin_pct}

【双平台差异】
- 平台A（内容电商）：依赖达人推荐和内容种草驱动销售，广告曝光前2名高曝光、3-4名中曝光、5-6名低曝光。佣金率通常较低。
- 平台B（货架电商）：用户主动搜索购买，广告竞价更集中（前1名高曝光、2-3名中曝光、4-6名低曝光）。佣金率通常较高但转化率高。

【重要提醒】
- 注意：本实验中平台策略每轮可能变化！请密切关注佣金率和补贴率的变动趋势。
- 每个平台的利润 = 该平台销售收入 - 生产成本 - 平台佣金 - 该平台广告费 - 达人佣金
- 广告费无论是否卖出都要支付，请谨慎投放
- 连续{EXIT_LOSE_ROUNDS}轮低于安全线将被迫退出该平台

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先进行策略分析，然后给出具体决策。
价格必须在{self.price_min}到{self.price_max}之间，广告必须在0到{self.ad_max}之间：
<thinking>你的策略分析（2-3句话）</thinking>
<decision>价格:X,广告A:Y,广告B:Z</decision>"""

    def decide_action(self, polA, polB):
        if self.exited_A:
            status_A = f"已退出平台A（冷却剩余{self.cooldown_A}轮）"
        elif self.last_sales_A > 0:
            status_A = f"毛利率{self.last_margin_A:.1%}（{'盈利' if self.last_margin_A > self.margin_safe else '低于安全线'}），销量{self.last_sales_A}件，连续亏损{self.consecutive_lose_A}轮"
        else:
            status_A = f"上轮无销量，连续亏损{self.consecutive_lose_A}轮"

        if self.exited_B:
            status_B = f"已退出平台B（冷却剩余{self.cooldown_B}轮）"
        elif self.last_sales_B > 0:
            status_B = f"毛利率{self.last_margin_B:.1%}（{'盈利' if self.last_margin_B > self.margin_safe else '低于安全线'}），销量{self.last_sales_B}件，连续亏损{self.consecutive_lose_B}轮"
        else:
            status_B = f"上轮无销量，连续亏损{self.consecutive_lose_B}轮"

        ctx = (f"平台A（内容电商）：佣金率={polA['commission']:.0%}，补贴率={polA['subsidy']:.0%}（用户实付打{1-polA['subsidy']:.0%}折）\n"
               f"  你在A的状态：{status_A}\n"
               f"平台B（货架电商）：佣金率={polB['commission']:.0%}，补贴率={polB['subsidy']:.0%}（用户实付打{1-polB['subsidy']:.0%}折）\n"
               f"  你在B的状态：{status_B}\n"
               f"同类竞品约{sum(1 for c in SELLER_CONFIGS if c[1]==self.seller_type)}家在两平台售卖。")

        active_on = []
        if not self.exited_A:
            active_on.append("A")
        if not self.exited_B:
            active_on.append("B")

        if not active_on:
            return {"price": 0, "ad_A": 0, "ad_B": 0, "thinking": "两平台均已退出"}

        task = (f"你当前在平台{'、'.join(active_on)}经营。请决定：\n"
                f"1. 商品定价（￥{self.price_min}~￥{self.price_max}，两平台同价）\n"
                f"2. 平台A广告投入（￥0~￥{self.ad_max}）{'——已退出，必须填0' if self.exited_A else ''}\n"
                f"3. 平台B广告投入（￥0~￥{self.ad_max}）{'——已退出，必须填0' if self.exited_B else ''}\n"
                f"注意：广告费无论是否卖出都会扣除。请考虑佣金率差异（A={polA['commission']:.0%} vs B={polB['commission']:.0%}）来分配投入。")

        dec, thinking = self.decide(ctx, task)

        # 解析价格（默认中位数）
        price = (self.price_min + self.price_max) / 2
        price_m = re.findall(r'价格[:：]?\s*(\d+\.?\d*)', dec)
        if price_m:
            val = float(price_m[0])
            price = max(self.price_min, min(self.price_max, val))
        else:
            nums = re.findall(r'\d+\.?\d*', dec)
            for n in nums:
                val = float(n)
                if self.price_min <= val <= self.price_max:
                    price = val
                    break

        ad_A = 0.0
        adA_m = re.findall(r'广告A[:：]?\s*(\d+\.?\d*)', dec)
        if adA_m and not self.exited_A:
            val = float(adA_m[0])
            ad_A = max(0, min(self.ad_max, val))

        ad_B = 0.0
        adB_m = re.findall(r'广告B[:：]?\s*(\d+\.?\d*)', dec)
        if adB_m and not self.exited_B:
            val = float(adB_m[0])
            ad_B = max(0, min(self.ad_max, val))

        if self.exited_A:
            ad_A = 0
        if self.exited_B:
            ad_B = 0

        return {"price": round(price, 1), "ad_A": round(ad_A, 1), "ad_B": round(ad_B, 1), "thinking": thinking}


# ================= 5. 达人Agent =================
INFLUENCER_CONFIGS = [
    ("Infu_T1", "种草型", 3, 0.06, "温和走心，重视商品品质与粉丝信任，偏好推荐中高端好物", "中高端"),
    ("Infu_T2", "种草型", 2, 0.06, "性价比种草达人，善于发现高性价比商品，均衡务实", "中端"),
    ("Infu_D1", "带货型", 4, 0.04, "冲量带货王，追求单场最大销量，偏好低价爆品走量", "低价"),
    ("Infu_D2", "带货型", 3, 0.04, "稳健带货，追求稳定的佣金收益，不冒进", "中端"),
    ("Infu_D3", "带货型", 5, 0.04, "头部达人，影响力大，偏好高客单价商品以获得更高佣金总额", "中高端"),
]


class InfluencerAgent(BaseAgent):
    def __init__(self, iid, flavor, inf_power, comm_rate, personality, preference):
        super().__init__(iid, "达人", f"{flavor}｜{personality}")
        self.flavor = flavor
        self.inf_power = inf_power
        self.comm_rate = comm_rate
        self.personality = personality
        self.preference = preference
        self.chosen_platform = "A"
        self.chosen_seller = None

    def reset(self):
        self.chosen_platform = "A"
        self.chosen_seller = None

    def build_prompt(self, memory, context, task):
        return f"""你是一个电商带货达人，需要在两个平台之间选择带货平台和合作商家。

【达人身份】
- 达人类型：{self.flavor}
- 性格特点：{self.personality}
- 带货影响力：{self.inf_power}/5
- 佣金比例：{self.comm_rate:.0%}
- 选品偏好：偏好{self.preference}商品

【双平台差异】
- 平台A（内容电商）：达人生态更成熟，你的影响力在A平台发挥更大
- 平台B（货架电商）：用户以搜索为主，达人推荐影响力较弱，但用户购买意图明确

【重要提醒】
- 注意：本实验中平台策略每轮可能变化！请关注佣金率和补贴率的变动。
- 你的佣金收入 = 你带来的销售额 × {self.comm_rate:.0%}

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

<thinking>你的分析（2-3句话）</thinking>
<decision>平台:A/B,商家:商家ID</decision>"""

    def choose_platform_and_seller(self, polA, polB, sellers_A_info, sellers_B_info):
        ctx = (f"平台A（内容电商）：佣金率={polA['commission']:.0%}，补贴率={polA['subsidy']:.0%}\n"
               f"平台A活跃商家：\n{sellers_A_info}\n\n"
               f"平台B（货架电商）：佣金率={polB['commission']:.0%}，补贴率={polB['subsidy']:.0%}\n"
               f"平台B活跃商家：\n{sellers_B_info}")
        task = (f"请选择：1.带货平台（A或B）；2.合作商家ID。你偏好{self.preference}类型的商品。\n"
                f"格式：<decision>平台:A/B,商家:商家ID</decision>")
        dec, thinking = self.decide(ctx, task)

        plat_match = re.search(r'平台[:：]?\s*([AB])', dec)
        if plat_match:
            self.chosen_platform = plat_match.group(1)
        elif "B" in dec and "A" not in dec:
            self.chosen_platform = "B"
        else:
            self.chosen_platform = "A"

        self.chosen_seller = None
        sids = re.findall(r'Seller_[A-Z]\d+', dec)
        if sids:
            self.chosen_seller = sids[0]

        return self.chosen_platform, self.chosen_seller, thinking


# ================= 6. 用户Agent =================
USER_CONFIGS = []
_low_personas = [
    "极度价格敏感，只买最便宜的商品，哪个平台便宜去哪个",
    "折扣猎手，哪个平台补贴高就去哪个平台",
    "容易被广告吸引，看到高曝光商品就想买",
    "信任达人推荐，达人说好就买，偏好平台A",
    "习惯使用平台A，不太愿意切换平台",
    "有品质意识但预算紧张，在预算内选最好的",
    "冲动型消费者，看到低价好物就下单",
    "理性比价型，会对比两个平台所有低价商品再决定",
    "回头客，倾向于回购之前买过的商家和平台",
    "新用户，好奇心强，两个平台都愿意尝试",
    "促销敏感型，补贴力度大的平台多买",
    "口碑导向，看重达人推荐和用户评价",
    "习惯购买型，偏好固定平台和商家",
    "社交推荐驱动，受达人影响大，偏向内容电商A",
    "随机浏览型，没有强偏好，两平台随机看看",
]
for i in range(15):
    USER_CONFIGS.append((f"User_L{i+1}", "低价敏感", 100, _low_personas[i]))

_mid_personas = [
    "重视达人推荐，达人背书的中端商品优先，偏好平台A",
    "品质优先，愿意为更好品质多付一些，两平台比较",
    "价格适中即可，不追求最低也不追求最贵",
    "理性消费者，几乎不看广告，自主判断，偏好搜索购物的平台B",
    "品牌忠诚型，认准的商家会持续复购",
    "尝鲜型，喜欢试新品和新商家，两平台都逛",
    "精打细算型，追求品质的同时关注性价比",
    "口碑驱动，看重达人和其他用户的推荐",
    "活动敏感，补贴活动期间倾向多买",
    "均衡考量型，综合考虑价格、品质、推荐、平台体验",
]
for i in range(10):
    USER_CONFIGS.append((f"User_M{i+1}", "品质性价比", 200, _mid_personas[i]))

_high_personas = [
    "只买高端精品，价格不是首要考虑因素，两平台都看",
    "深度信任头部达人推荐的高端商品，偏好平台A",
    "纯品质导向，只关注产品本身的品质，偏好搜索比较",
    "品牌忠诚度极高，只买认可的高端商家",
    "自主决策型，不受广告和达人影响，独立判断",
]
for i in range(5):
    USER_CONFIGS.append((f"User_H{i+1}", "高端品质", 400, _high_personas[i]))


class UserAgent(BaseAgent):
    def __init__(self, uid, user_type, budget, persona):
        super().__init__(uid, "用户", f"{user_type}｜{persona}")
        self.user_type = user_type
        self.budget = budget
        self.persona = persona

    def build_prompt(self, memory, context, task):
        return ""

    def build_purchase_prompt(self, memory, context, task, influenced_by_inf, influenced_by_ad):
        if self.user_type == "低价敏感":
            type_guide = (
                "作为低价敏感用户，你优先选择最便宜的商品，不管在哪个平台。"
                "中端和高端商品通常超出你的预算，除非补贴后价格在你预算内，否则不要购买。"
                "哪个平台补贴力度大、商品便宜，你就倾向于在那个平台购买。"
            )
        elif self.user_type == "品质性价比":
            type_guide = (
                "作为品质性价比用户，你追求品质与价格的平衡。"
                "你倾向于购买中端商品，低价商品品质可能不够，高端商品超出你的预算考量。"
                "你会在两个平台之间比较，选择性价比最高的商品。"
            )
        else:
            type_guide = (
                "作为高端品质用户，你追求最好的产品品质。"
                "你倾向于购买高端或中高端商品，低价商品基本不在你的考虑范围内。"
                "价格不是你的首要顾虑，品质和品牌才是。"
            )

        inf_desc = "本轮你看到了达人推荐内容，达人推荐会影响你的判断（达人主要活跃在平台A）" if influenced_by_inf else "本轮你没有关注达人推荐，更依赖自己的判断"
        ad_desc = "本轮你注意到了广告展示，高曝光商品吸引了你的注意" if influenced_by_ad else "本轮你对广告不敏感，广告曝光不影响你的选择"

        return f"""你是一个可以在两个电商平台（A和B）购物的用户，需要决定在哪个平台买什么商品。

【用户身份】
- 用户类型：{self.user_type}
- 性格特点：{self.persona}
- 本轮预算上限：￥{self.budget}（你绝对不能购买单价超过￥{self.budget}的商品！）

【双平台差异】
- 平台A（内容电商）：有达人推荐和内容种草，购物体验更社交化
- 平台B（货架电商）：搜索购买为主，商品展示更直接

【选购指导】
{type_guide}

【本轮状态】
- {inf_desc}
- {ad_desc}

【重要约束】
- 你最多购买2件商品（可来自不同平台），也可以选择不买
- 每件商品的实付价格必须 ≤ 你的预算￥{self.budget}，超过预算的商品禁止购买

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

<thinking>你的分析（考虑预算、偏好、平台差异）</thinking>
<decision>平台X-商家ID1,平台Y-商家ID2</decision> 或 <decision>不买</decision>"""

    def decide_purchase(self, items_A_info, items_B_info, reco_info, influenced_by_inf, influenced_by_ad):
        memory = self.read_memory()
        ctx = (f"【平台A（内容电商）商品】（价格为补贴后实付价格）\n{items_A_info}\n\n"
               f"【平台B（货架电商）商品】（价格为补贴后实付价格）\n{items_B_info}\n\n"
               f"【达人推荐信息】\n{reco_info}")
        task = (f"浏览两个平台后选择购买（最多2件，可来自不同平台，也可不买）。预算￥{self.budget}，不能买超过这个价格的商品。\n"
                f"格式：<decision>平台X-商家ID1,平台Y-商家ID2</decision> 或 <decision>不买</decision>")
        prompt = self.build_purchase_prompt(memory, ctx, task, influenced_by_inf, influenced_by_ad)
        res = generate_llm_response(prompt)

        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{ctx[:80]}...｜决策：{decision}")

        purchases = []
        if "不买" in decision or "none" in decision.lower():
            return purchases, thinking

        pairs = re.findall(r'([AB])\s*[-–]\s*(Seller_[A-Z]\d+)', decision)
        if pairs:
            for plat, sid in pairs[:2]:
                purchases.append((plat, sid))
        else:
            sids = re.findall(r'Seller_[A-Z]\d+', decision)
            for sid in sids[:2]:
                purchases.append(("A", sid))

        return purchases, thinking


# ================= 7. 广告曝光分配 =================
def assign_exposure_A(seller_ads):
    sorted_s = sorted([(s, a) for s, a in seller_ads.items() if a > 0], key=lambda x: x[1], reverse=True)
    exp = {}
    for i, (sid, _) in enumerate(sorted_s):
        if i < 2: exp[sid] = "高曝光"
        elif i < 4: exp[sid] = "中曝光"
        elif i < 6: exp[sid] = "低曝光"
    return exp


def assign_exposure_B(seller_ads):
    sorted_s = sorted([(s, a) for s, a in seller_ads.items() if a > 0], key=lambda x: x[1], reverse=True)
    exp = {}
    for i, (sid, _) in enumerate(sorted_s):
        if i < 1: exp[sid] = "高曝光"
        elif i < 3: exp[sid] = "中曝光"
        elif i < 6: exp[sid] = "低曝光"
    return exp


def build_items_info(active_sellers, seller_actions, ad_key, exposure_map, influencer_reco_map, subsidy_rate):
    lines = []
    for s in active_sellers:
        sid = s.agent_id
        act = seller_actions.get(sid, {"price": 0, ad_key: 0})
        if act["price"] == 0:
            continue
        exp = exposure_map.get(sid, "无")
        inf_tag = influencer_reco_map.get(sid, "")
        original_price = act["price"]
        actual_price = round(original_price * (1 - subsidy_rate), 1)
        line = f"  {sid}({s.seller_type}) 实付￥{actual_price}（原价￥{original_price}，补贴{subsidy_rate:.0%}）"
        if exp != "无":
            line += f" 【{exp}】"
        if inf_tag:
            line += f" {inf_tag}"
        lines.append(line)
    return "\n".join(lines) if lines else "  暂无商品"


# ================= 8. 仿真主逻辑 =================
def run_dynamic_experiment(init_strategy, exp_label):
    print(f"\n{'='*60}")
    print(f"实验组 {exp_label}: A初始佣{init_strategy['A_comm']:.2f}补{init_strategy['A_sub']:.2f} | B初始佣{init_strategy['B_comm']:.2f}补{init_strategy['B_sub']:.2f}")
    print(f"{'='*60}")

    mem_dir = os.path.join(MEMORY_DIR, exp_label)
    os.makedirs(mem_dir, exist_ok=True)

    # 初始化平台
    platA = PlatformAgent("A", A_COMM_RANGE, A_SUB_RANGE)
    platA.memory_file = os.path.join(mem_dir, "Platform_A_memory.txt")
    platA.init_memory()
    platA.reset(init_strategy["A_comm"], init_strategy["A_sub"])

    platB = PlatformAgent("B", B_COMM_RANGE, B_SUB_RANGE)
    platB.memory_file = os.path.join(mem_dir, "Platform_B_memory.txt")
    platB.init_memory()
    platB.reset(init_strategy["B_comm"], init_strategy["B_sub"])

    sellers = []
    for cfg in SELLER_CONFIGS:
        s = SellerAgent(*cfg)
        s.memory_file = os.path.join(mem_dir, f"{s.agent_id}_memory.txt")
        s.init_memory()
        s.reset()
        sellers.append(s)

    influencers = []
    for cfg in INFLUENCER_CONFIGS:
        inf = InfluencerAgent(*cfg)
        inf.memory_file = os.path.join(mem_dir, f"{inf.agent_id}_memory.txt")
        inf.init_memory()
        inf.reset()
        influencers.append(inf)

    users = []
    for cfg in USER_CONFIGS:
        u = UserAgent(*cfg)
        u.memory_file = os.path.join(mem_dir, f"{u.agent_id}_memory.txt")
        u.init_memory()
        users.append(u)

    round_records = []
    platform_records = []
    seller_records = []
    influencer_records = []
    user_records = []
    transaction_records = []

    no_change_count = 0
    converged = False

    for rnd in range(1, MAX_ROUNDS + 1):
        print(f"\n--- {exp_label} 第{rnd}轮 ---")

        # ---- 平台策略决策 ----
        if rnd == 1:
            polA = {"commission": platA.current_comm, "subsidy": platA.current_sub}
            polB = {"commission": platB.current_comm, "subsidy": platB.current_sub}
            a_strategy = {"commission": polA["commission"], "subsidy": polA["subsidy"],
                         "old_commission": polA["commission"], "old_subsidy": polA["subsidy"],
                         "thinking": "第一轮使用初始策略"}
            b_strategy = {"commission": polB["commission"], "subsidy": polB["subsidy"],
                         "old_commission": polB["commission"], "old_subsidy": polB["subsidy"],
                         "thinking": "第一轮使用初始策略"}
        else:
            a_strategy = platA.decide_strategy(platB.current_comm, platB.current_sub, platB.last_gmv, platB.last_revenue, rnd)
            b_strategy = platB.decide_strategy(platA.current_comm, platA.current_sub, platA.last_gmv, platA.last_revenue, rnd)
            polA = {"commission": a_strategy["commission"], "subsidy": a_strategy["subsidy"]}
            polB = {"commission": b_strategy["commission"], "subsidy": b_strategy["subsidy"]}

        print(f"  A策略: 佣{polA['commission']:.3f} 补{polA['subsidy']:.3f} | B策略: 佣{polB['commission']:.3f} 补{polB['subsidy']:.3f}")

        platform_records.append({
            "exp": exp_label, "round": rnd, "platform": "A",
            "commission": polA["commission"], "subsidy": polA["subsidy"],
            "old_commission": a_strategy["old_commission"], "old_subsidy": a_strategy["old_subsidy"],
            "comm_change": abs(polA["commission"] - a_strategy["old_commission"]),
            "sub_change": abs(polA["subsidy"] - a_strategy["old_subsidy"]),
            "thinking": a_strategy["thinking"]
        })
        platform_records.append({
            "exp": exp_label, "round": rnd, "platform": "B",
            "commission": polB["commission"], "subsidy": polB["subsidy"],
            "old_commission": b_strategy["old_commission"], "old_subsidy": b_strategy["old_subsidy"],
            "comm_change": abs(polB["commission"] - b_strategy["old_commission"]),
            "sub_change": abs(polB["subsidy"] - b_strategy["old_subsidy"]),
            "thinking": b_strategy["thinking"]
        })

        # ---- 冷却处理 ----
        for s in sellers:
            if s.exited_A and s.cooldown_A > 0:
                s.cooldown_A -= 1
                if s.cooldown_A == 0:
                    s.exited_A = False
                    s.consecutive_lose_A = 0
                    print(f"  {s.agent_id} 平台A冷却结束")
            if s.exited_B and s.cooldown_B > 0:
                s.cooldown_B -= 1
                if s.cooldown_B == 0:
                    s.exited_B = False
                    s.consecutive_lose_B = 0
                    print(f"  {s.agent_id} 平台B冷却结束")

        # ---- 商家决策 ----
        sellers_on_A = [s for s in sellers if not s.exited_A]
        sellers_on_B = [s for s in sellers if not s.exited_B]
        all_active = [s for s in sellers if not s.exited_A or not s.exited_B]
        seller_actions = {}
        for s in all_active:
            act = s.decide_action(polA, polB)
            seller_actions[s.agent_id] = act
            print(f"  {s.agent_id}: 价格={act['price']}, 广告A={act['ad_A']}, 广告B={act['ad_B']}")

        for s in sellers:
            if s.agent_id not in seller_actions:
                seller_actions[s.agent_id] = {"price": 0, "ad_A": 0, "ad_B": 0, "thinking": "两平台均已退出"}

        # ---- 广告曝光 ----
        ads_A = {s.agent_id: seller_actions[s.agent_id]["ad_A"] for s in sellers_on_A}
        ads_B = {s.agent_id: seller_actions[s.agent_id]["ad_B"] for s in sellers_on_B}
        exposure_A = assign_exposure_A(ads_A)
        exposure_B = assign_exposure_B(ads_B)

        # ---- 达人选平台+商家 ----
        sellers_A_info = "\n".join([
            f"  {s.agent_id}({s.seller_type}) 定价￥{seller_actions[s.agent_id]['price']} 广告投入￥{seller_actions[s.agent_id]['ad_A']}"
            for s in sellers_on_A if seller_actions[s.agent_id]["price"] > 0
        ])
        sellers_B_info = "\n".join([
            f"  {s.agent_id}({s.seller_type}) 定价￥{seller_actions[s.agent_id]['price']} 广告投入￥{seller_actions[s.agent_id]['ad_B']}"
            for s in sellers_on_B if seller_actions[s.agent_id]["price"] > 0
        ])

        influencer_reco_A = {}
        influencer_reco_B = {}
        influencer_chosen = {}
        inf_round_data = []

        for inf in influencers:
            plat, sid, inf_think = inf.choose_platform_and_seller(polA, polB, sellers_A_info, sellers_B_info)
            tag = f"【达人推荐:{inf.agent_id}(影响力{inf.inf_power}/5)】"
            if sid and sid in seller_actions and seller_actions[sid]["price"] > 0:
                if plat == "A":
                    influencer_reco_A[sid] = influencer_reco_A.get(sid, "") + " " + tag
                else:
                    influencer_reco_B[sid] = influencer_reco_B.get(sid, "") + " " + tag
                influencer_chosen[inf.agent_id] = (plat, sid)

            inf_round_data.append({
                "exp": exp_label, "round": rnd,
                "influencer_id": inf.agent_id, "flavor": inf.flavor,
                "inf_power": inf.inf_power, "comm_rate": inf.comm_rate,
                "chosen_platform": plat, "chosen_seller": sid or "无",
                "thinking": inf_think
            })

        # ---- 构建商品展示（补贴后实付价） ----
        items_A_info = build_items_info(sellers_on_A, seller_actions, "ad_A", exposure_A, influencer_reco_A, polA["subsidy"])
        items_B_info = build_items_info(sellers_on_B, seller_actions, "ad_B", exposure_B, influencer_reco_B, polB["subsidy"])
        all_reco = {}
        for sid, tag in influencer_reco_A.items():
            all_reco[f"A-{sid}"] = tag
        for sid, tag in influencer_reco_B.items():
            all_reco[f"B-{sid}"] = tag
        reco_info = "\n".join([f"  {k}: {v}" for k, v in all_reco.items()]) if all_reco else "  无达人推荐"

        # 反向映射
        seller_to_influencers = {}
        for inf_id, (plat, sid) in influencer_chosen.items():
            seller_to_influencers.setdefault((plat, sid), []).append(inf_id)
        inf_recommended_A = {sid for (plat, sid) in influencer_chosen.values() if plat == "A"}
        inf_recommended_B = {sid for (plat, sid) in influencer_chosen.values() if plat == "B"}

        # ---- 用户购买 ----
        round_transactions = []
        user_round_data = []

        for u in users:
            influenced_by_inf = random.choice([0, 1])
            influenced_by_ad = random.choice([0, 1])

            purchases, u_think = u.decide_purchase(
                items_A_info, items_B_info, reco_info,
                influenced_by_inf, influenced_by_ad
            )

            # ★ 后处理硬过滤
            valid_purchases = []
            for plat, sid in purchases:
                if sid in seller_actions and seller_actions[sid]["price"] > 0:
                    sub_rate = polA["subsidy"] if plat == "A" else polB["subsidy"]
                    actual_price = round(seller_actions[sid]["price"] * (1 - sub_rate), 1)
                    if actual_price <= u.budget:
                        valid_purchases.append((plat, sid))
                    else:
                        print(f"    ⛔ {u.agent_id} 试图在{plat}购买 {sid}(￥{actual_price})超预算￥{u.budget}，已拦截")

            if not valid_purchases:
                user_round_data.append({
                    "exp": exp_label, "round": rnd,
                    "user_id": u.agent_id, "user_type": u.user_type,
                    "persona": u.persona, "budget": u.budget,
                    "influenced_by_influencer": influenced_by_inf,
                    "influenced_by_ad": influenced_by_ad,
                    "purchased": 0, "purchase_platform": "",
                    "seller_bought": "", "price_paid": 0,
                    "is_influencer_driven": False,
                    "thinking": u_think
                })
            else:
                for plat, sid in valid_purchases:
                    price = seller_actions[sid]["price"]
                    sub_rate = polA["subsidy"] if plat == "A" else polB["subsidy"]
                    actual = round(price * (1 - sub_rate), 1)
                    if plat == "A":
                        is_inf_driven = (influenced_by_inf == 1 and sid in inf_recommended_A)
                    else:
                        is_inf_driven = (influenced_by_inf == 1 and sid in inf_recommended_B)

                    round_transactions.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "seller_id": sid, "platform": plat,
                        "original_price": price, "subsidy_rate": sub_rate,
                        "actual_price": actual,
                        "is_influencer_driven": is_inf_driven
                    })
                    user_round_data.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "persona": u.persona, "budget": u.budget,
                        "influenced_by_influencer": influenced_by_inf,
                        "influenced_by_ad": influenced_by_ad,
                        "purchased": 1, "purchase_platform": plat,
                        "seller_bought": sid, "price_paid": actual,
                        "is_influencer_driven": is_inf_driven,
                        "thinking": u_think
                    })

        # ---- 清算 ----
        seller_sales_A = {s.agent_id: 0 for s in sellers}
        seller_sales_B = {s.agent_id: 0 for s in sellers}
        seller_gmv_A = {s.agent_id: 0.0 for s in sellers}
        seller_gmv_B = {s.agent_id: 0.0 for s in sellers}
        seller_inf_driven_gmv_A = {s.agent_id: 0.0 for s in sellers}
        seller_inf_driven_gmv_B = {s.agent_id: 0.0 for s in sellers}

        for tx in round_transactions:
            sid = tx["seller_id"]
            if tx["platform"] == "A":
                seller_sales_A[sid] += 1
                seller_gmv_A[sid] += tx["original_price"]
                if tx["is_influencer_driven"]:
                    seller_inf_driven_gmv_A[sid] += tx["original_price"]
            else:
                seller_sales_B[sid] += 1
                seller_gmv_B[sid] += tx["original_price"]
                if tx["is_influencer_driven"]:
                    seller_inf_driven_gmv_B[sid] += tx["original_price"]

        # ★ 达人佣金仅基于达人实际带来的订单
        seller_inf_cost_A = {s.agent_id: 0.0 for s in sellers}
        seller_inf_cost_B = {s.agent_id: 0.0 for s in sellers}
        for inf in influencers:
            if inf.agent_id in influencer_chosen:
                plat, sid = influencer_chosen[inf.agent_id]
                num_infs = len(seller_to_influencers.get((plat, sid), []))
                if num_infs > 0:
                    if plat == "A":
                        share_gmv = seller_inf_driven_gmv_A[sid] / num_infs
                        seller_inf_cost_A[sid] += share_gmv * inf.comm_rate
                    else:
                        share_gmv = seller_inf_driven_gmv_B[sid] / num_infs
                        seller_inf_cost_B[sid] += share_gmv * inf.comm_rate

        total_gmv_A = sum(seller_gmv_A.values())
        total_gmv_B = sum(seller_gmv_B.values())

        # ---- 商家清算 ----
        for s in sellers:
            sid = s.agent_id
            act = seller_actions[sid]

            # === 平台A ===
            gmvA = seller_gmv_A[sid]
            salesA = seller_sales_A[sid]
            adA = act["ad_A"]

            if not s.exited_A and sid in [x.agent_id for x in sellers_on_A]:
                commA = gmvA * polA["commission"]
                costA = s.unit_cost * salesA
                infA = seller_inf_cost_A[sid]
                # ★ 方案A + 广告费始终计入
                profitA = gmvA - costA - commA - adA - infA
                marginA = profitA / gmvA if gmvA > 0 else (-adA if adA > 0 else 0)
                s.last_margin_A = marginA
                s.last_sales_A = salesA

                if gmvA == 0 and adA > 0:
                    s.consecutive_lose_A += 1
                elif gmvA > 0 and marginA < s.margin_safe:
                    s.consecutive_lose_A += 1
                elif gmvA > 0:
                    s.consecutive_lose_A = 0

                if s.consecutive_lose_A >= EXIT_LOSE_ROUNDS and not s.exited_A:
                    s.exited_A = True
                    s.cooldown_A = COOLDOWN_ROUNDS
                    print(f"  ⚠ {sid} 退出平台A，冷却{COOLDOWN_ROUNDS}轮")
            else:
                commA = costA = infA = 0
                profitA = marginA = 0
                salesA = gmvA = adA = 0

            # === 平台B ===
            gmvB = seller_gmv_B[sid]
            salesB = seller_sales_B[sid]
            adB = act["ad_B"]

            if not s.exited_B and sid in [x.agent_id for x in sellers_on_B]:
                commB = gmvB * polB["commission"]
                costB = s.unit_cost * salesB
                infB = seller_inf_cost_B[sid]
                profitB = gmvB - costB - commB - adB - infB
                marginB = profitB / gmvB if gmvB > 0 else (-adB if adB > 0 else 0)
                s.last_margin_B = marginB
                s.last_sales_B = salesB

                if gmvB == 0 and adB > 0:
                    s.consecutive_lose_B += 1
                elif gmvB > 0 and marginB < s.margin_safe:
                    s.consecutive_lose_B += 1
                elif gmvB > 0:
                    s.consecutive_lose_B = 0

                if s.consecutive_lose_B >= EXIT_LOSE_ROUNDS and not s.exited_B:
                    s.exited_B = True
                    s.cooldown_B = COOLDOWN_ROUNDS
                    print(f"  ⚠ {sid} 退出平台B，冷却{COOLDOWN_ROUNDS}轮")
            else:
                commB = costB = infB = 0
                profitB = marginB = 0
                salesB = gmvB = adB = 0

            seller_records.append({
                "exp": exp_label, "round": rnd,
                "seller_id": sid, "seller_type": s.seller_type, "character": s.character,
                "unit_cost": s.unit_cost, "price": act["price"],
                "ad_A": act["ad_A"], "ad_B": act["ad_B"],
                "exposure_A": exposure_A.get(sid, "无"), "exposure_B": exposure_B.get(sid, "无"),
                "sales_A": salesA, "gmv_A": round(gmvA, 2),
                "comm_cost_A": round(commA, 2), "prod_cost_A": round(costA, 2),
                "inf_cost_A": round(infA, 2), "inf_driven_gmv_A": round(seller_inf_driven_gmv_A[sid], 2),
                "ad_cost_A": round(adA, 2), "profit_A": round(profitA, 2), "margin_A": round(marginA, 4),
                "sales_B": salesB, "gmv_B": round(gmvB, 2),
                "comm_cost_B": round(commB, 2), "prod_cost_B": round(costB, 2),
                "inf_cost_B": round(infB, 2), "inf_driven_gmv_B": round(seller_inf_driven_gmv_B[sid], 2),
                "ad_cost_B": round(adB, 2), "profit_B": round(profitB, 2), "margin_B": round(marginB, 4),
                "consecutive_lose_A": s.consecutive_lose_A, "exited_A": s.exited_A, "cooldown_A": s.cooldown_A,
                "consecutive_lose_B": s.consecutive_lose_B, "exited_B": s.exited_B, "cooldown_B": s.cooldown_B,
                "thinking": act["thinking"]
            })

        # ---- 更新平台指标 ----
        # ★ 平台营收 = 佣金收入 + 广告收入（所有活跃商家） - 补贴支出（方案A）
        plat_comm_A = sum(seller_gmv_A[s.agent_id] * polA["commission"] for s in sellers)
        plat_sub_A = sum(seller_gmv_A[s.agent_id] * polA["subsidy"] for s in sellers)
        plat_ad_A = sum(seller_actions[s.agent_id]["ad_A"] for s in sellers_on_A)
        plat_rev_A = plat_comm_A - plat_sub_A + plat_ad_A

        plat_comm_B = sum(seller_gmv_B[s.agent_id] * polB["commission"] for s in sellers)
        plat_sub_B = sum(seller_gmv_B[s.agent_id] * polB["subsidy"] for s in sellers)
        plat_ad_B = sum(seller_actions[s.agent_id]["ad_B"] for s in sellers_on_B)
        plat_rev_B = plat_comm_B - plat_sub_B + plat_ad_B

        total_gmv = total_gmv_A + total_gmv_B
        platA.last_gmv = total_gmv_A
        platA.last_revenue = plat_rev_A
        platA.last_market_share = total_gmv_A / total_gmv if total_gmv > 0 else 0.5
        platA.last_active_sellers = len(sellers_on_A)
        platA.last_exited_sellers = sum(1 for s in sellers if s.exited_A)
        platA.last_transactions = sum(1 for tx in round_transactions if tx["platform"] == "A")
        platA.last_inf_count = sum(1 for inf in influencers if inf.chosen_platform == "A")

        platB.last_gmv = total_gmv_B
        platB.last_revenue = plat_rev_B
        platB.last_market_share = total_gmv_B / total_gmv if total_gmv > 0 else 0.5
        platB.last_active_sellers = len(sellers_on_B)
        platB.last_exited_sellers = sum(1 for s in sellers if s.exited_B)
        platB.last_transactions = sum(1 for tx in round_transactions if tx["platform"] == "B")
        platB.last_inf_count = sum(1 for inf in influencers if inf.chosen_platform == "B")

        inf_driven_tx = sum(1 for tx in round_transactions if tx["is_influencer_driven"])

        round_records.append({
            "exp": exp_label, "round": rnd,
            "A_comm": polA["commission"], "A_sub": polA["subsidy"],
            "B_comm": polB["commission"], "B_sub": polB["subsidy"],
            "gmv_A": round(total_gmv_A, 2), "gmv_B": round(total_gmv_B, 2),
            "market_share_A": round(platA.last_market_share, 4),
            "transactions": len(round_transactions),
            "inf_driven_transactions": inf_driven_tx,
            "active_sellers_A": len(sellers_on_A), "active_sellers_B": len(sellers_on_B),
            "exited_sellers_A": platA.last_exited_sellers, "exited_sellers_B": platB.last_exited_sellers,
            "plat_comm_income_A": round(plat_comm_A, 2), "plat_sub_expense_A": round(plat_sub_A, 2),
            "plat_ad_income_A": round(plat_ad_A, 2), "plat_revenue_A": round(plat_rev_A, 2),
            "plat_comm_income_B": round(plat_comm_B, 2), "plat_sub_expense_B": round(plat_sub_B, 2),
            "plat_ad_income_B": round(plat_ad_B, 2), "plat_revenue_B": round(plat_rev_B, 2),
            "influencers_on_A": platA.last_inf_count,
            "influencers_on_B": platB.last_inf_count,
            "users_purchased": sum(1 for ud in user_round_data if ud.get("purchased") == 1),
            "converged": False
        })

        influencer_records.extend(inf_round_data)
        user_records.extend(user_round_data)
        transaction_records.extend(round_transactions)

        print(f"  A: GMV={total_gmv_A:.0f} 营收={plat_rev_A:.0f} 份额={platA.last_market_share:.1%}")
        print(f"  B: GMV={total_gmv_B:.0f} 营收={plat_rev_B:.0f} 份额={platB.last_market_share:.1%}")

        # ---- 收敛判断 ----
        if rnd > 1:
            a_comm_chg = abs(polA["commission"] - a_strategy["old_commission"])
            a_sub_chg = abs(polA["subsidy"] - a_strategy["old_subsidy"])
            b_comm_chg = abs(polB["commission"] - b_strategy["old_commission"])
            b_sub_chg = abs(polB["subsidy"] - b_strategy["old_subsidy"])

            if (a_comm_chg < CONVERGENCE_THRESHOLD and a_sub_chg < CONVERGENCE_THRESHOLD and
                b_comm_chg < CONVERGENCE_THRESHOLD and b_sub_chg < CONVERGENCE_THRESHOLD):
                no_change_count += 1
                print(f"  📊 策略基本未变 ({no_change_count}/{CONVERGENCE_ROUNDS})")
            else:
                no_change_count = 0

            if no_change_count >= CONVERGENCE_ROUNDS:
                converged = True
                round_records[-1]["converged"] = True
                print(f"  ✅ 达到收敛条件！连续{CONVERGENCE_ROUNDS}轮策略未显著变化")
                break

    return round_records, platform_records, seller_records, influencer_records, user_records, transaction_records, converged


# ================= 9. 运行8组实验 =================
all_round = []
all_platform = []
all_seller = []
all_influencer = []
all_user = []
all_transaction = []

for idx, init_strat in enumerate(INIT_STRATEGIES):
    label = f"Init{idx+1}_AC{init_strat['A_comm']:.2f}_AS{init_strat['A_sub']:.2f}_BC{init_strat['B_comm']:.2f}_BS{init_strat['B_sub']:.2f}"
    rr, pr, sr, ir, ur, tr, conv = run_dynamic_experiment(init_strat, label)
    all_round.extend(rr)
    all_platform.extend(pr)
    all_seller.extend(sr)
    all_influencer.extend(ir)
    all_user.extend(ur)
    all_transaction.extend(tr)

    # 每组保存
    pd.DataFrame(all_round).to_csv(f"{RESULTS_DIR}/exp3_round_summary.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_platform).to_csv(f"{RESULTS_DIR}/exp3_platform_strategy.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_seller).to_csv(f"{RESULTS_DIR}/exp3_seller_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_influencer).to_csv(f"{RESULTS_DIR}/exp3_influencer_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_user).to_csv(f"{RESULTS_DIR}/exp3_user_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_transaction).to_csv(f"{RESULTS_DIR}/exp3_transactions.csv", index=False, encoding="utf-8-sig")

    conv_str = "已收敛" if conv else "未收敛(达到最大轮次)"
    print(f"\n✅ 实验组 {label} ({idx+1}/8) 完成 [{conv_str}]")

print(f"\n{'='*60}")
print(f"✅ 实验三全部完成！共8组实验，数据保存在 {RESULTS_DIR}")
print(f"{'='*60}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
加载模型: Qwen/Qwen2.5-14B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型加载完成

实验组 Init1_AC0.06_AS0.01_BC0.12_BS0.04: A初始佣0.06补0.01 | B初始佣0.12补0.04

--- Init1_AC0.06_AS0.01_BC0.12_BS0.04 第1轮 ---
  A策略: 佣0.060 补0.010 | B策略: 佣0.120 补0.040
  Seller_L1: 价格=50.0, 广告A=1.0, 广告B=1.0
  Seller_L2: 价格=48.0, 广告A=2.0, 广告B=1.0
  Seller_L3: 价格=47.0, 广告A=1.0, 广告B=3
  Seller_L4: 价格=48.0, 广告A=1.0, 广告B=0.5
  Seller_L5: 价格=47.0, 广告A=2.0, 广告B=1.0
  Seller_L6: 价格=50.0, 广告A=3, 广告B=1.0
  Seller_M1: 价格=150.0, 广告A=10.0, 广告B=5.0
  Seller_M2: 价格=150.0, 广告A=5.0, 广告B=10.0
  Seller_M3: 价格=150.0, 广告A=10.0, 广告B=5.0
  Seller_H1: 价格=285.0, 广告A=35.0, 广告B=15.0
    ⛔ User_M5 试图在A购买 Seller_H1(￥282.1)超预算￥200，已拦截
  A: GMV=2350 营收=188 份额=49.3%
  B: GMV=2413 营收=236 份额=50.7%

--- Init1_AC0.06_AS0.01_BC0.12_BS0.04 第2轮 ---
  A策略: 佣0.060 补0.030 | B策略: 佣0.120 补0.050
  Seller_L1: 价格=48.0, 广告A=2.0, 广告B=3
  Seller_L2: 价格=49.0, 广告A=1.0, 广告B=3
  Seller_L3: 价格=47.0, 广告A=2.0, 广告B=2.0
  Seller_L4: 价格=48.0, 广告A=0, 广告B=1.0
  Seller_L5: 价格=46.0, 广告A=3, 广告B=1.0
  Seller_L6: 价格=53.0, 广告A=3, 广告B=2.0
  Seller_M1: 价格=